# Phase 2, step 2: E2 and E3 on the INSTANCE test split

| Exp | Trained on | Tested on | Where |
|---|---|---|---|
| E2 | STEAD (your 5 models, seed 99) | INSTANCE test split | `RUN_EVAL`, CPU is enough |
| E3 | INSTANCE train split | INSTANCE test split | `RUN_E3_SMOKE`, `RUN_E3_FULL` (GPU), one bottleneck per session |

**Datasets to attach:** `prabhanshu22/instance-mixtures-snr40` (always), `prabhanshu22/wdn-runs-seed99` (E2 and the final tables), and for E3 the repo is cloned (Internet ON).
**Sessions:** (1) `RUN_EVAL = True`, GPU not needed. (2) smoke + full training for `None`, save output as a dataset `wdn-runs-instance-none`. (3) same for `attention`. (4) final `RUN_EVAL = True` with all three run datasets attached.

The training script is `step3_training_1D.py` of Phase 1 plus ONE option, `--presplit`, which uses the split stored in the data (split by earthquake before mixing). Hyperparameters are unchanged (batch 128, 100 epochs, float64, Adam 1e-3 with warm-up, no early stopping). Note: 18,000 training rows give about 6 times fewer weight updates per epoch than the 112,200 of Phase 1; this is a consequence of keeping the epochs fixed and goes into the deviation log.

In [ ]:
# SWITCHES
RUN_EVAL = True          # Evaluation Purposes
RUN_E3_SMOKE = False     # 2-epoch smoke run on 5000 evenly spaced rows, prints the projected time
RUN_E3_FULL = False      # full training of BOTTLENECK on the INSTANCE train split

BOTTLENECKS = ["None", "Linear", "LSTM", "attention", "Transformer"]
E3_BOTTLENECK = "LSTM"
SEED = 99
EPOCHS = 100
SESSION_LIMIT_H = 12.0
SAFETY_FACTOR = 0.9
FORCE_FULL_RUN = False

EPOCHS_REQUIRED = 100
C_RECOMPUTE = False
EV_CLIP = -1.0
BANDPASS_HZ = (0.5, 4.9)
N_BOOT = 1000
SNR_BINS_PER_DECADE = 4
FAIL_FRACTION = 0.02
BOOT_SEED = 12345
EXAMPLE_SEED = 7
EXTRA_SEARCH_ROOTS = []
INSTANCE_FILE = "training_datasets_INSTANCE_snr40.hdf5"

In [2]:
import os, sys, glob, json, time, shutil, subprocess, platform, datetime, re, traceback, hashlib
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle/input")
BASE = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
RESULTS, LOGS, FIGS, RUNS_OUT = BASE/"results", BASE/"logs", BASE/"figures", BASE/"wdn_runs"
for d in (RESULTS, LOGS, FIGS, RUNS_OUT):
    d.mkdir(parents=True, exist_ok=True)
CODE_DIR = BASE/"WaveDecompNet-paper"
MODEL_DIR = BASE/"Model_and_datasets_1D_all_snr_40_unshuffled_equal_epoch100"
REPO_URL = "https://github.com/yinjiuxun/WaveDecompNet-paper"
EXPECTED_COMMIT = "afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1"
SEARCH_ROOTS = (["/kaggle/input"] if ON_KAGGLE else []) + [str(p) for p in EXTRA_SEARCH_ROOTS]

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(map(str, cmd))}\n{r.stdout}\n{r.stderr}")
    return r.stdout.strip()

print("python", platform.python_version())
import numpy as np
print("numpy", np.__version__)
try:
    import torch
    print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0), "| memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
except Exception as e:
    print("torch import failed:", e)
print("disk free GB (working dir):", round(shutil.disk_usage(BASE).free / 1e9, 1))

def ensure(pip_name, module=None):
    try:
        __import__(module or pip_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
ensure("h5py"); ensure("pandas"); ensure("scikit-learn", "sklearn"); ensure("scipy")

if not (CODE_DIR/".git").exists():
    sh(["git", "clone", "--depth", "1", REPO_URL, str(CODE_DIR)])
GIT_HASH = sh(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"])
if GIT_HASH != EXPECTED_COMMIT:
    sh(["git", "-C", str(CODE_DIR), "fetch", "--depth", "1", "origin", EXPECTED_COMMIT])
    sh(["git", "-C", str(CODE_DIR), "checkout", EXPECTED_COMMIT])
    GIT_HASH = sh(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"])
assert GIT_HASH == EXPECTED_COMMIT, f"repo commit {GIT_HASH} differs from {EXPECTED_COMMIT}"
print("repo commit:", GIT_HASH)

def log_entry(part, command, outcome, seed=None):
    p = LOGS/"lab_log.md"
    if not p.exists():
        p.write_text("# Lab log\n\n| date (UTC) | part | command | commit | seed | outcome |\n|---|---|---|---|---|---|\n")
    now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M")
    with open(p, "a") as f:
        f.write(f"| {now} | {part} | `{command}` | {GIT_HASH[:10]} | {seed if seed is not None else ''} | {outcome} |\n")

def find_files(name):
    out = []
    for r in SEARCH_ROOTS:
        out += glob.glob(f"{r}/**/{name}", recursive=True)
    return sorted(out)

def find_one(name, required=True):
    hits = find_files(name)
    if not hits and required:
        raise FileNotFoundError(f"{name} not found. Attach the dataset that contains it.")
    if len(hits) > 1:
        print(f"WARNING: {name} found {len(hits)} times, using {hits[0]}")
    return Path(hits[0]) if hits else None

python 3.13.16
numpy 2.1.3
torch 2.11.0+cu130 | cuda available: True
GPU: Tesla T4 | memory GB: 15.6
disk free GB (working dir): 20.9
repo commit: afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1


In [ ]:
CONFIG_PY = r'''"""Central configuration for the WaveDecompNet replication (works locally and on Kaggle).

Every other script imports this file, so paths and seeds live in exactly one place.
Override the STEAD location with the environment variable WDN_STEAD_DIR if needed.
"""
import glob
import os

ON_KAGGLE = os.path.exists("/kaggle/input")


def _find_stead(name):
    env = os.environ.get("WDN_STEAD_DIR")
    if env:
        return os.path.join(env, name)
    if ON_KAGGLE:
        hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        if hits:
            return hits[0]
    return os.path.join(os.getcwd(), "stead", name)

# ---------------------------------------------------------------- paths
WORK_DIR = "/kaggle/working" if ON_KAGGLE else os.getcwd()

STEAD_HDF5 = _find_stead("merged.hdf5")
STEAD_CSV = _find_stead("merged.csv")

DATASET_DIR = os.path.join(WORK_DIR, "training_datasets")
POHA_DIR = os.path.join(WORK_DIR, "continuous_waveforms")
POHA_MSEED = os.path.join(POHA_DIR, "IU.POHA.00.20210601-20210801.mseed")

STEAD_MIX_H5 = os.path.join(DATASET_DIR, "training_datasets_STEAD_waveform_snr_40.hdf5")
POHA_MIX_H5 = os.path.join(DATASET_DIR, "training_datasets_STEAD_plus_POHA_snr_40_unshuffled.hdf5")
# paper-faithful name (the file is actually shuffled WITH replacement, see step2d)
FINAL_DATASET = os.path.join(DATASET_DIR, "training_datasets_all_snr_40_unshuffled.hdf5")
FINAL_DATASET_PERMUTED = os.path.join(DATASET_DIR, "training_datasets_all_snr_40_permuted.hdf5")

MODEL_DIR = os.path.join(WORK_DIR, "Model_and_datasets_1D_all_snr_40_unshuffled_equal_epoch100")

# ---------------------------------------------------------------- data recipe (from the repo)
SNR_MIN_DB = 40
N_EVENTS = 100000                 # step2b: number of STEAD mixtures
F_DOWNSAMPLE = 10                 # Hz
N_APPEND = 1000                   # chunk size when writing HDF5
STEAD_SEEDS = dict(earthquake=101, noise=102, shift=103, snr=104)   # step2b
POHA_SEEDS = dict(earthquake=111, noise=112, shift=113, snr=114)    # step2c
POHA_AMPLITUDE_THRESHOLD = 6      # step2c: drop samples above 6 x median amplitude
POHA_EXPECTED_FS = 40.0           # step2c calls decimate(4) -> 10 Hz, so it assumes 40 Hz

# step2c in the original code writes only full chunks of N_APPEND, so the last
# (N mod 1000) windows are silently dropped. False = keep that behaviour (fidelity).
KEEP_REMAINDER = False

# step2d in the original code shuffles with np.random.choice(N, N): WITH replacement,
# no seed. True = same behaviour but seeded and logged; False = a proper permutation.
SHUFFLE_WITH_REPLACEMENT = True
SHUFFLE_SEED = 0

# ---------------------------------------------------------------- training (step3)
BATCH_SIZE = 128
EPOCHS = 100
LR = 1e-3
MIN_EPOCHS = 30
PATIENCE = None                   # None = no early stopping (always 100 epochs)
TORCH_SEED = 99
SPLIT_SEED_1 = 13
SPLIT_SEED_2 = 20
TRAIN_SIZE = 0.6
TEST_SIZE = 0.5                   # of the remaining 40 % -> 20 % val, 20 % test
'''

STEP3_PRESPLIT_PY = r'''"""step3_presplit = step3 (patched) + ONE new option: --presplit.

--presplit uses the dataset split stored in the data file (dataset `split_id`: 0 = train, 1 = validation, 2 = test) instead of the
random 60/20/20 split after loading. Needed for the INSTANCE data, which is split by earthquake BEFORE mixing. Without --presplit the
behaviour is identical to step3_training_1D.py.

Original docstring:
step3 (patched): train ONE bottleneck per run (needed for Kaggle's session limits).

Same model, loss, optimiser, LR schedule, split (60/20/20, seeds 13 and 20) and seeds as the repo.
Changes: command-line arguments, paths from config.py, no `tabnanny` import, no `verbose=` in
LambdaLR (removed in newer PyTorch), figure saved headless, run info saved as JSON.

Quick smoke test first (a few minutes):
    python step3_training_1D.py --bottleneck None --epochs 2 --subset 5000 --tag _smoke
Full run:
    python step3_training_1D.py --bottleneck LSTM
"""
import argparse
import copy
import json
import os
import random
import time

import h5py
import matplotlib
matplotlib.use("Agg")
import numpy as np
import torch
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

import config as C
from autoencoder_1D_models_torch import SeismogramDecoder, SeismogramEncoder, SeisSeparator
from torch_tools import WaveformDataset, training_loop_branches, try_gpu

BOTTLENECKS = ["None", "Linear", "LSTM", "attention", "Transformer"]


def make_bottleneck(name):
    if name == "None":
        return None
    if name == "Linear":
        return torch.nn.Linear(64, 64, dtype=torch.float64)
    if name == "LSTM":
        return torch.nn.LSTM(64, 32, 2, bidirectional=True, batch_first=True, dtype=torch.float64)
    if name == "attention":
        return torch.nn.MultiheadAttention(64, 4, 0.2, batch_first=True, dtype=torch.float64)
    if name == "Transformer":
        layer = torch.nn.TransformerEncoderLayer(d_model=64, nhead=4, dtype=torch.float64)
        return torch.nn.TransformerEncoder(layer, num_layers=1)
    raise NameError("bottleneck type has not been defined!")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--bottleneck", required=True, choices=BOTTLENECKS)
    p.add_argument("--data", default=C.FINAL_DATASET)
    p.add_argument("--seed", type=int, default=C.TORCH_SEED)
    p.add_argument("--epochs", type=int, default=C.EPOCHS)
    p.add_argument("--subset", type=int, default=None, help="use only the first N samples (smoke test)")
    p.add_argument("--tag", default=None, help="suffix of the output folder")
    p.add_argument("--presplit", action="store_true", help="use split_id stored in the data file (0 train, 1 val, 2 test)")
    args = p.parse_args()

    tag = args.tag if args.tag is not None else ("_warmup" if args.seed == C.TORCH_SEED else f"_warmup_seed{args.seed}")
    model_structure = "Branch_Encoder_Decoder"
    model_name = f"{model_structure}_{args.bottleneck}"
    out_dir = os.path.join(C.MODEL_DIR, model_name + tag)
    os.makedirs(out_dir, exist_ok=True)
    device = try_gpu()
    print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

    # ---- data
    print("#" * 12 + " Loading data " + "#" * 12)
    if args.presplit:
        with h5py.File(args.data, "r") as f:
            n_all = f["X_train"].shape[0]
            assert "split_id" in f, "--presplit needs a split_id dataset in the data file"
            sid_all = f["split_id"][:]
            if args.subset is None or args.subset >= n_all:
                sel = np.arange(n_all)
                X_train, Y_train = f["X_train"][:], f["Y_train"][:]
            else:                                    # evenly spaced rows keep the train / val / test proportions
                sel = np.unique(np.linspace(0, n_all - 1, args.subset).astype(int))
                X_train, Y_train = f["X_train"][sel], f["Y_train"][sel]
            sid = sid_all[sel]
        print("data:", X_train.shape, X_train.dtype, "| presplit rows train/val/test:", [int((sid == k).sum()) for k in range(3)])
        assert (sid == 0).any() and (sid == 1).any(), "no train or validation rows selected"
        X_training, Y_training = X_train[sid == 0], Y_train[sid == 0]
        X_validate, Y_validate = X_train[sid == 1], Y_train[sid == 1]
        del X_train, Y_train
    else:
        with h5py.File(args.data, "r") as f:
            n = f["X_train"].shape[0] if args.subset is None else min(args.subset, f["X_train"].shape[0])
            X_train, Y_train = f["X_train"][:n], f["Y_train"][:n]
        print("data:", X_train.shape, X_train.dtype)

        X_training, X_test, Y_training, Y_test = train_test_split(
            X_train, Y_train, train_size=C.TRAIN_SIZE, random_state=C.SPLIT_SEED_1)
        X_validate, X_test, Y_validate, Y_test = train_test_split(
            X_test, Y_test, test_size=C.TEST_SIZE, random_state=C.SPLIT_SEED_2)
        del X_train, Y_train, X_test, Y_test

    # ---- seeds (same as the repo, torch seed configurable for multi-seed runs)
    torch.manual_seed(args.seed)
    random.seed(0)
    np.random.seed(20)
    torch.backends.cudnn.benchmark = False

    training_data = WaveformDataset(X_training, Y_training)
    validate_data = WaveformDataset(X_validate, Y_validate)

    # ---- model
    bottleneck = make_bottleneck(args.bottleneck)
    encoder = SeismogramEncoder()
    decoder_earthquake = SeismogramDecoder(bottleneck=copy.deepcopy(bottleneck))
    decoder_noise = SeismogramDecoder(bottleneck=copy.deepcopy(bottleneck))
    model = SeisSeparator(model_name, encoder, decoder_earthquake, decoder_noise).to(device=device)
    n_params = int(sum(p.numel() for p in model.parameters()))
    print(f"{model_name}: {n_params} parameters")

    batch_size, lr = C.BATCH_SIZE, C.LR
    loss_fn = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    LR_func = lambda epoch: (epoch + 1) / 11 if (epoch <= 10) else (0.95 ** epoch if (epoch < 50) else 0.95 ** 50)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=LR_func)

    train_iter = DataLoader(training_data, batch_size=batch_size, shuffle=True)
    validate_iter = DataLoader(validate_data, batch_size=batch_size, shuffle=False)

    # ---- train
    print("#" * 12 + " training model " + model_name + " " + "#" * 12)
    t0 = time.time()
    model, avg_train_losses, avg_valid_losses, partial_loss = training_loop_branches(
        train_iter, validate_iter, model, loss_fn, optimizer, scheduler,
        epochs=args.epochs, patience=C.PATIENCE, device=device, minimum_epochs=C.MIN_EPOCHS)
    runtime = time.time() - t0
    print(f"Training is done! {runtime / 60:.1f} min")

    # ---- save
    torch.save(model, os.path.join(out_dir, f"{model_name}_Model.pth"))
    with h5py.File(os.path.join(out_dir, f"{model_name}_Training_history.hdf5"), "w") as f:
        f.create_dataset("loss", data=avg_train_losses)
        f.create_dataset("val_loss", data=avg_valid_losses)
        f.create_dataset("earthquake_loss", data=partial_loss[0])
        f.create_dataset("earthquake_val_loss", data=partial_loss[1])
        f.create_dataset("noise_loss", data=partial_loss[2])
        f.create_dataset("noise_val_loss", data=partial_loss[3])
    with h5py.File(os.path.join(out_dir, f"{model_name}_Dataset_split.hdf5"), "w") as f:
        f.attrs["model_name"] = model_name
        f.attrs["train_size"] = C.TRAIN_SIZE
        f.attrs["test_size"] = C.TEST_SIZE
        f.attrs["rand_seed1"] = C.SPLIT_SEED_1
        f.attrs["rand_seed2"] = C.SPLIT_SEED_2
        f.attrs["presplit"] = bool(args.presplit)
    with open(os.path.join(out_dir, "run_info.json"), "w") as f:
        json.dump(dict(model_name=model_name, bottleneck=args.bottleneck, torch_seed=args.seed,
                       epochs=args.epochs, subset=args.subset, data=args.data, presplit=bool(args.presplit),
                       n_train=len(training_data), n_val=len(validate_data), n_params=n_params,
                       runtime_s=runtime, device=str(device),
                       gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else None,
                       torch=torch.__version__, final_train_loss=float(avg_train_losses[-1]),
                       final_val_loss=float(avg_valid_losses[-1])), f, indent=2)

    plt.figure()
    plt.plot(avg_train_losses, "o", label="loss")
    plt.plot(avg_valid_losses, "-", label="Validation loss")
    for k, (lab, mk) in enumerate(zip(["earthquake train loss", "earthquake valid loss",
                                       "noise train loss", "noise valid loss"], ["o", "", "o", ""])):
        plt.plot(partial_loss[k], marker=mk, label=lab)
    plt.yscale("log")
    plt.legend()
    plt.title(model_name)
    plt.savefig(os.path.join(out_dir, f"{model_name}_Training_history.png"))
    plt.close()
    print("saved to", out_dir)


if __name__ == "__main__":
    main()
'''

(CODE_DIR/"config.py").write_text(CONFIG_PY)
(CODE_DIR/"step3_training_1D_presplit.py").write_text(STEP3_PRESPLIT_PY)
SCRIPT_SHA = {n: hashlib.sha256((CODE_DIR/n).read_bytes()).hexdigest()[:16] for n in ("config.py", "step3_training_1D_presplit.py")}
print("scripts written, sha256[:16]:", SCRIPT_SHA)
tt = CODE_DIR/"torch_tools.py"; txt = tt.read_text()
if "np.Inf" in txt:
    tt.write_text(txt.replace("np.Inf", "np.inf")); print("patched np.Inf -> np.inf in torch_tools.py")
assert "np.Inf" not in tt.read_text()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

scripts written, sha256[:16]: {'config.py': '146bcb7aa227c74e', 'step3_training_1D_presplit.py': 'd634fc8c73dd624d'}
patched np.Inf -> np.inf in torch_tools.py


In [ ]:
import pandas as pd
import h5py

import pandas as pd

def read_csv_keep(path):
    """pandas treats the string 'None' (our bottleneck name) as missing by default; keep it."""
    return pd.read_csv(path, keep_default_na=False, na_values=[""])

def save_fig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGS/f"{name}.{ext}", bbox_inches="tight", dpi=200)

def prepare_torch():
    """Import torch and the repo modules; disable the eval fast path so eval == the training-time math."""
    import importlib, torch
    if str(CODE_DIR) not in sys.path:
        sys.path.insert(0, str(CODE_DIR))
    mods = {n: importlib.import_module(n) for n in ("autoencoder_1D_models_torch", "torch_tools", "utilities")}
    try:
        torch.backends.mha.set_fastpath_enabled(False)
    except Exception:
        pass
    return torch, mods

COMPAT_DEFAULTS = {"norm_first": False, "activation_relu_or_gelu": 1, "enable_nested_tensor": False,
                   "use_nested_tensor": False, "mask_check": True}
COMPAT_LOG = []

def load_model(path, device):
    """Load a whole pickled model (torch.save(model)). Needs autoencoder_1D_models_torch importable."""
    torch, _ = prepare_torch()
    try:
        m = torch.load(str(path), map_location=device, weights_only=False)
    except TypeError:
        m = torch.load(str(path), map_location=device)
    return m.to(device).double().eval()

def check_forward(model, device):
    """Dummy forward pass. Optional compat shim ONLY if an old pickle lacks an attribute newer torch expects
    (defaults = what the constructor sets). Every use is recorded in COMPAT_LOG and goes into the deviation log."""
    torch, _ = prepare_torch()
    x = torch.randn(2, 3, 600, dtype=torch.float64, device=device)
    for _ in range(8):
        try:
            with torch.no_grad():
                o = model(x)
            assert len(o) == 2 and tuple(o[0].shape) == (2, 3, 600) and tuple(o[1].shape) == (2, 3, 600), "unexpected output shapes"
            return True, None
        except AttributeError as e:
            name = (re.search(r"has no attribute '(\w+)'", str(e)) or [None, None])[1]
            if name not in COMPAT_DEFAULTS:
                return False, traceback.format_exc()
            for mod in model.modules():
                if type(mod).__name__ in ("TransformerEncoderLayer", "TransformerEncoder") and not hasattr(mod, name):
                    setattr(mod, name, COMPAT_DEFAULTS[name])
            COMPAT_LOG.append(name)
        except Exception:
            return False, traceback.format_exc()
    return False, "compat shim did not converge: " + ", ".join(COMPAT_LOG)

def predict(model, Xt, device, bs=256):
    """Xt: (N, 3, 600) float64 -> (earthquake, noise), both (N, 3, 600)."""
    torch, _ = prepare_torch()
    pe = np.empty_like(Xt); pn = np.empty_like(Xt)
    with torch.no_grad():
        for i in range(0, len(Xt), bs):
            xb = torch.from_numpy(Xt[i:i + bs]).to(device)
            o1, o2 = model(xb)
            pe[i:i + bs] = o1.cpu().numpy(); pn[i:i + bs] = o2.cpu().numpy()
    return pe, pn

def get_device():
    torch, _ = prepare_torch()
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")



def ev_score(y, p, axis=-1):
    """Explained variance exactly as sklearn.explained_variance_score(multioutput='raw_values') (the definition used in step5),
    vectorised: one value per trace and component, time on `axis`."""
    d = y - p
    num = np.mean((d - d.mean(axis, keepdims=True)) ** 2, axis=axis)
    den = np.mean((y - y.mean(axis, keepdims=True)) ** 2, axis=axis)
    out = np.ones_like(num)
    ok = (num != 0) & (den != 0)
    out[ok] = 1 - num[ok] / den[ok]
    out[(num != 0) & (den == 0)] = 0.0
    return out

def step5_snr_db(signal, noise, axis=-1):
    """signal_to_noise_ratio() of step5, in dB, per component."""
    s0 = np.sum(signal ** 2, axis=axis) / (np.sum(noise ** 2, axis=axis) + 1e-6)
    return 10 * np.log10(s0 + 1e-6)

def pearson_rows(a, b):
    a = a.reshape(len(a), -1); b = b.reshape(len(b), -1)
    a = a - a.mean(1, keepdims=True); b = b - b.mean(1, keepdims=True)
    den = np.sqrt((a ** 2).sum(1) * (b ** 2).sum(1))
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(den > 0, (a * b).sum(1) / den, np.nan)

def metrics_chunked(Xt, Yt, pe, pn, chunk=4000):
    """Per-sample metrics. Xt, Yt, pe, pn: (N, 3, 600). Earthquake truth = Yt, noise truth = Xt - Yt (as in training)."""
    cols = {k: [] for k in ("ev_eq", "ev_nz", "snr5", "mse_eq", "mse_noise", "corr")}
    for i in range(0, len(Xt), chunk):
        x, y, a, b = Xt[i:i + chunk], Yt[i:i + chunk], pe[i:i + chunk], pn[i:i + chunk]
        tn = x - y
        cols["ev_eq"].append(ev_score(y, a)); cols["ev_nz"].append(ev_score(tn, b))
        cols["snr5"].append(step5_snr_db(y, tn))
        cols["mse_eq"].append(((y - a) ** 2).mean((1, 2))); cols["mse_noise"].append(((tn - b) ** 2).mean((1, 2)))
        cols["corr"].append(pearson_rows(y, a))
    return {k: np.concatenate(v) for k, v in cols.items()}

def boot_ci_mean(x, n_boot=1000, rng=None, alpha=0.05, chunk=100):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    rng = rng or np.random.default_rng(BOOT_SEED)
    means = np.concatenate([x[rng.integers(0, len(x), (min(chunk, n_boot - i), len(x)))].mean(1) for i in range(0, n_boot, chunk)])
    return float(x.mean()), float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

def boot_ci_median(x, n_boot=1000, rng=None, alpha=0.05, chunk=100):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    rng = rng or np.random.default_rng(BOOT_SEED)
    med = np.concatenate([np.median(x[rng.integers(0, len(x), (min(chunk, n_boot - i), len(x)))], axis=1) for i in range(0, n_boot, chunk)])
    return float(np.median(x)), float(np.quantile(med, alpha / 2)), float(np.quantile(med, 1 - alpha / 2))

def snr_edges(lo=0.1, hi=100.0, per_decade=4):
    return np.logspace(np.log10(lo), np.log10(hi), int(round(np.log10(hi / lo) * per_decade)) + 1)

def bin_index(snr, edges):
    b = np.searchsorted(edges, snr, side="right") - 1
    b[snr == edges[-1]] = len(edges) - 2
    b[(snr < edges[0]) | (snr > edges[-1]) | ~np.isfinite(snr)] = -1
    return b

def summarise_model(df):
    """Model-level numbers. Pooled medians follow step5 (every trace-component is one value)."""
    n5 = (len(df) // 256) * 256            # step5 drops the last incomplete batch of 256
    def pool(d, p): return d[[f"{p}_c0", f"{p}_c1", f"{p}_c2"]].to_numpy().ravel()
    e, nz = pool(df, "ev_eq"), pool(df, "ev_noise"); d5 = df.iloc[:n5]
    return dict(n_samples=len(df),
                median_ev_eq=float(np.median(e)), median_ev_noise=float(np.median(nz)),
                mean_ev_eq_clipped=float(np.clip(e, EV_CLIP, 1).mean()), mean_ev_noise_clipped=float(np.clip(nz, EV_CLIP, 1).mean()),
                frac_ev_eq_lt0=float((e < 0).mean()), frac_ev_noise_lt0=float((nz < 0).mean()),
                test_loss=float(df["mse_total"].mean()), mean_mse_eq=float(df["mse_eq"].mean()), mean_mse_noise=float(df["mse_noise"].mean()),
                mean_corr_eq=float(np.nanmean(df["corr_eq"])),
                median_ev_eq_step5subset=float(np.median(pool(d5, "ev_eq"))), median_ev_noise_step5subset=float(np.median(pool(d5, "ev_noise"))),
                test_loss_step5subset=float(d5["mse_total"].mean()))

In [ ]:
DATA_PATH = find_one(INSTANCE_FILE, required=False)
print("INSTANCE mixtures:", DATA_PATH)
META = None; FACTS = {}
if DATA_PATH is not None:
    with h5py.File(DATA_PATH, "r") as f:
        for k in f:
            print(f"  {k:20s} {f[k].shape} {f[k].dtype}")
        print("  attrs:", {k: (v.item() if hasattr(v, "item") else v) for k, v in f.attrs.items()})
        assert f["X_train"].shape == f["Y_train"].shape and f["X_train"].shape[1:] == (600, 3)
        sid = f["split_id"][:]
        assert (np.diff(sid.astype(int)) >= 0).all(), "rows are expected in the order train | val | test"
        dec = lambda a: np.array([x.decode() if isinstance(x, (bytes, np.bytes_)) else str(x) for x in a], dtype=object)
        META = pd.DataFrame(dict(file_row=np.arange(len(sid)), split_id=sid, source_id=f["source_id"][:], earthquake_name=dec(f["earthquake_name"][:]),
                                 noise_name=dec(f["noise_name"][:]), noise_window=f["noise_window"][:], station_channels=dec(f["station_channels"][:]),
                                 source_magnitude=f["source_magnitude"][:], event_snr_mean_db=f["event_snr_mean_db"][:], snr=f["snr"][:],
                                 shift=f["shift"][:], quake_energy_frac=f["quake_energy_frac"][:], p_time_in_window_s=f["p_time_in_window_s"][:]))
        te = np.where(sid == 2)[0]; assert te[-1] - te[0] + 1 == len(te), "test rows are not contiguous"
        t0 = time.time(); Xte = f["X_train"][te[0]:te[-1] + 1]; Yte = f["Y_train"][te[0]:te[-1] + 1]
    MTEST = META.iloc[te].reset_index(drop=True); MTEST.insert(0, "row", np.arange(len(MTEST)))
    Xt = np.ascontiguousarray(np.moveaxis(Xte, 1, -1)); Yt = np.ascontiguousarray(np.moveaxis(Yte, 1, -1)); del Xte, Yte
    FACTS = dict(n_rows=int(len(sid)), n_train=int((sid == 0).sum()), n_val=int((sid == 1).sum()), n_test=int(len(te)),
                 test_earthquakes=int(MTEST.source_id.nunique()), test_earthquake_traces=int(MTEST.earthquake_name.nunique()),
                 test_noise_windows=int(len(set(zip(MTEST.noise_name, MTEST.noise_window)))),
                 test_frac_energy_lt_1pct=float((MTEST.quake_energy_frac < 0.01).mean()),
                 test_channels=MTEST.station_channels.value_counts().to_dict())
    json.dump(FACTS, open(RESULTS/"instance_data_facts.json", "w"), indent=2, default=str)
    MTEST.to_csv(RESULTS/"instance_test_metadata.csv.gz", index=False)
    print("test split loaded:", Xt.shape, f"{time.time() - t0:.0f} s")
    print(json.dumps(FACTS, indent=1, default=str))
else:
    print("STOP: attach prabhanshu22/instance-mixtures-snr40. Only the training part can not run without it.")

INSTANCE mixtures: /kaggle/input/datasets/prabhanshu22/instance-mixtures-snr40/training_datasets_INSTANCE_snr40.hdf5
  X_train              (30000, 600, 3) float64
  Y_train              (30000, 600, 3) float64
  earthquake_name      (30000,) object
  event_snr_mean_db    (30000,) float64
  noise_name           (30000,) object
  noise_window         (30000,) int64
  p_time_in_window_s   (30000,) float64
  quake_energy_frac    (30000,) float64
  shift                (30000,) float64
  snr                  (30000,) float64
  source_id            (30000,) int64
  source_magnitude     (30000,) float64
  split_id             (30000,) int8
  station_channels     (30000,) object
  time                 (600,) float64
  attrs: {'component_order': 'E,N,Z as stored (axes [0, 1, 2])', 'n_test': 6000, 'n_train': 18000, 'n_val': 6000, 'noise_windows': 2, 'pre_p_s': 10.0, 'snr_min_db': 40.0, 'split_seed_1': 13, 'split_seed_2': 20, 'units': 'raw counts (earthquake and noise)'}
test split loaded: (6000

## Main Training Loop

In [6]:
def run_tag(seed):
    return "_instance" if seed == 99 else f"_instance_seed{seed}"

SMOKE = {}
def run_step3(bottleneck, seed, epochs, subset=None, tag=None, log_name="train.log"):
    cmd = [sys.executable, "-u", "step3_training_1D_presplit.py", "--presplit", "--bottleneck", bottleneck, "--data", str(DATA_PATH),
           "--seed", str(seed), "--epochs", str(epochs)]
    if subset: cmd += ["--subset", str(subset)]
    if tag is not None: cmd += ["--tag", tag]
    print(" ".join(cmd)); t0 = time.time(); lines = []
    proc = subprocess.Popen(cmd, cwd=CODE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with open(LOGS/log_name, "w") as lf:
        for line in proc.stdout:
            print(line, end=""); lf.write(line); lines.append(line)
    return proc.wait(), "".join(lines), time.time() - t0

def gpu_warning():
    try:
        import torch
        if not torch.cuda.is_available():
            print("WARNING: no GPU visible. Turn on a GPU accelerator in the notebook settings."); return
        n = torch.cuda.get_device_name(0); print("GPU:", n)
        if "T4" in n:
            print("NOTE: T4 has slow float64 (this model is float64). Read the smoke projection before a full run.")
    except Exception:
        pass

if RUN_E3_SMOKE:
    assert DATA_PATH is not None, "attach the INSTANCE mixtures"
    gpu_warning()
    rc, out, dt = run_step3("None", 99, 2, subset=5000, tag="_instance_smoke", log_name="smoke_instance.log")
    assert rc == 0, "smoke run failed, read the traceback above"
    times = [float(x) for x in re.findall(r"time per epoch: ([\d.]+) s", out)]
    info = json.load(open(MODEL_DIR/"Branch_Encoder_Decoder_None_instance_smoke"/"run_info.json"))
    scale = (FACTS["n_train"] + FACTS["n_val"]) / (info["n_train"] + info["n_val"])
    proj_h = times[-1] * scale * EPOCHS / 3600
    SMOKE.update(epoch_times=times, projected_hours=proj_h, ok=proj_h <= SAFETY_FACTOR * SESSION_LIMIT_H)
    print(f"\nsmoke epoch times (s): {times}; scale to full train+val x{scale:.1f}")
    print(f"PROJECTED time for {EPOCHS} epochs of 'None': {proj_h:.2f} h (session limit {SESSION_LIMIT_H} h). attention / LSTM / Transformer are slower.")
    print("OK to start the full run" if SMOKE["ok"] else "STOP: projected time exceeds the session limit.")
    json.dump(SMOKE, open(RESULTS/"smoke_projection_instance.json", "w"), indent=2)
    log_entry("E3 smoke", "step3_presplit --bottleneck None --epochs 2 --subset 5000", f"epoch times {times}, projected {proj_h:.2f} h", 99)
else:
    print("smoke run skipped")

if RUN_E3_FULL:
    assert DATA_PATH is not None, "attach the INSTANCE mixtures"
    if not SMOKE.get("ok") and not FORCE_FULL_RUN:
        raise RuntimeError("Run the smoke test first in this session (RUN_E3_SMOKE) or set FORCE_FULL_RUN = True.")
    gpu_warning()
    rc, out, dt = run_step3(E3_BOTTLENECK, SEED, EPOCHS, tag=run_tag(SEED), log_name=f"train_instance_{E3_BOTTLENECK}_seed{SEED}.log")
    assert rc == 0, "training failed, read the traceback above"
    folder = f"Branch_Encoder_Decoder_{E3_BOTTLENECK}{run_tag(SEED)}"; src_dir = MODEL_DIR/folder
    info = json.load(open(src_dir/"run_info.json"))
    shutil.copytree(src_dir, RUNS_OUT/folder, dirs_exist_ok=True)
    cfg = dict(git_hash=GIT_HASH, seed=SEED, bottleneck=E3_BOTTLENECK, dataset=str(info.get("data")), presplit=True, torch=info.get("torch"), gpu=info.get("gpu"),
               numpy=np.__version__, python=platform.python_version(), script_sha=SCRIPT_SHA, date_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
    (RUNS_OUT/folder/f"config_{info['model_name']}.json").write_text(json.dumps(cfg, indent=2))
    size_mb = sum(p.stat().st_size for p in (RUNS_OUT/folder).rglob("*")) / 1e6
    print(f"\nDONE {folder}: {info['runtime_s']/60:.1f} min, {info['n_params']} params, final val loss {info['final_val_loss']:.5f}, n_train {info['n_train']}, n_val {info['n_val']}")
    print(f"copied to {RUNS_OUT/folder} ({size_mb:.1f} MB). Save a version, then make a dataset from wdn_runs/.")
    log_entry("E3 full", f"step3_presplit --bottleneck {E3_BOTTLENECK} --seed {SEED} --epochs {EPOCHS}", f"{info['runtime_s']/60:.1f} min, val loss {info['final_val_loss']:.5f}", SEED)
else:
    print("full training skipped")

smoke run skipped
full training skipped


## Evaluation 

In [7]:
def discover_runs2():
    """All finished full runs under the search roots and the working dir. trained_on = INSTANCE if the data path says so."""
    found = {}
    for root in SEARCH_ROOTS + [str(BASE)]:
        for p in sorted(glob.glob(f"{root}/**/Branch_Encoder_Decoder_*/run_info.json", recursive=True)):
            try: info = json.load(open(p))
            except Exception: continue
            if info.get("subset") is not None or info.get("epochs") != EPOCHS_REQUIRED: continue
            d = Path(p).parent; mf = d/f"{info['model_name']}_Model.pth"
            if not mf.exists(): continue
            on = "instance" if "INSTANCE" in str(info.get("data", "")).upper() else "stead"
            key = (on, info["bottleneck"], int(info["torch_seed"]))
            if key not in found: found[key] = dict(dir=d, model_file=mf, info=info)
    return found

def discover_models():
    models = []
    for (on, b, s), r in sorted(discover_runs2().items()):
        i = r["info"]
        models.append(dict(id=f"{on}_{b}_seed{s}", kind=on, bottleneck=b, seed=s, path=r["model_file"], n_params=i.get("n_params"),
                           runtime_s=i.get("runtime_s"), gpu=i.get("gpu"), torch=i.get("torch"),
                           final_train_loss=i.get("final_train_loss"), final_val_loss=i.get("final_val_loss")))
    for b in BOTTLENECKS:
        d = CODE_DIR/"trained_models"/f"Branch_Encoder_Decoder_{b}"
        models.append(dict(id=f"pretrained_{b}", kind="pretrained", bottleneck=b, seed=-1, path=d/f"Branch_Encoder_Decoder_{b}_Model.pth"))
    return models

REG_PATH = RESULTS/"model_registry_instance.csv"
def load_registry():
    if not REG_PATH.exists(): return {}
    df = read_csv_keep(REG_PATH); df["bottleneck"] = df["bottleneck"].fillna("")
    return df.set_index("model_id").to_dict("index")
def save_registry(reg):
    pd.DataFrame([dict(model_id=k, **v) for k, v in reg.items()]).to_csv(REG_PATH, index=False)

def per_sample_frame(meta, m):
    df = meta[["row", "file_row", "source_id", "earthquake_name", "noise_name", "noise_window", "station_channels", "source_magnitude",
               "event_snr_mean_db", "snr", "shift", "quake_energy_frac", "p_time_in_window_s"]].copy()
    for c in range(3): df[f"ev_eq_c{c}"] = m["ev_eq"][:, c]
    df["ev_eq_mean_clipped"] = np.clip(m["ev_eq"], EV_CLIP, 1).mean(1)
    for c in range(3): df[f"ev_noise_c{c}"] = m["ev_nz"][:, c]
    df["ev_noise_mean_clipped"] = np.clip(m["ev_nz"], EV_CLIP, 1).mean(1)
    df["mse_eq"] = m["mse_eq"]; df["mse_noise"] = m["mse_noise"]; df["mse_total"] = m["mse_eq"] + m["mse_noise"]
    df["corr_eq"] = m["corr"]
    for c in range(3): df[f"snr_step5_db_c{c}"] = m["snr5"][:, c]
    return df

if RUN_EVAL:
    assert DATA_PATH is not None, "attach the INSTANCE mixtures"
    torch, mods = prepare_torch(); dev = get_device(); REG = load_registry(); errors = []
    def finish(mid, pe, pn, info):
        df = per_sample_frame(MTEST, metrics_chunked(Xt, Yt, pe, pn))
        df.to_csv(RESULTS/f"per_sample_{mid}.csv.gz", index=False)
        REG[mid] = {**REG.get(mid, {}), **info, "status": "ok"}; save_registry(REG)
    for m in discover_models():
        out = RESULTS/f"per_sample_{m['id']}.csv.gz"
        if out.exists() and not C_RECOMPUTE:
            print("reuse", m["id"]); continue
        try:
            model = load_model(m["path"], dev); ok, err = check_forward(model, dev)
            if not ok: raise RuntimeError(err)
            pe, pn = predict(model, Xt, dev)
            npar = int(sum(p.numel() for p in model.parameters()))
            info = {k: m.get(k) for k in ("kind", "bottleneck", "seed", "runtime_s", "gpu", "torch", "final_train_loss", "final_val_loss")}
            finish(m["id"], pe, pn, {**info, "n_params": npar, "model_path": str(m["path"])})
            print("evaluated", m["id"], f"({npar} params)"); del model, pe, pn
        except Exception:
            errors.append(m["id"]); REG[m["id"]] = dict(kind=m["kind"], bottleneck=m["bottleneck"], seed=m["seed"], status="FAILED")
            save_registry(REG); print("FAILED", m["id"]); print(traceback.format_exc())
    from scipy.signal import butter, sosfiltfilt
    sos = butter(4, list(BANDPASS_HZ), btype="band", fs=10, output="sos"); bp = sosfiltfilt(sos, Xt, axis=-1)
    for mid, pe, pn, desc in (("baseline_identity", Xt, Xt, "earthquake output = input, noise output = input"),
                               ("baseline_bandpass", bp, Xt - bp, f"zero-phase 4th-order Butterworth band-pass {BANDPASS_HZ} Hz; noise = input - filtered")):
        if (RESULTS/f"per_sample_{mid}.csv.gz").exists() and not C_RECOMPUTE: continue
        finish(mid, pe, pn, dict(kind="baseline", bottleneck=mid.replace("baseline_", ""), seed=-1, n_params=0, note=desc)); print("baseline", mid)
    print("failed models:", errors, "| compat shim used for:", COMPAT_LOG)
    log_entry("E2/E3 eval", "evaluate STEAD-trained, INSTANCE-trained, pretrained and baselines on the INSTANCE test split", f"failed={errors}, shim={COMPAT_LOG}")
else:
    print("evaluation skipped")

evaluated instance_LSTM_seed99 (178442 params)
evaluated instance_Linear_seed99 (86410 params)
evaluated instance_None_seed99 (78090 params)
evaluated instance_Transformer_seed99 (640394 params)
evaluated instance_attention_seed99 (111370 params)
evaluated stead_LSTM_seed99 (178442 params)
evaluated stead_Linear_seed99 (86410 params)
evaluated stead_None_seed99 (78090 params)
evaluated stead_Transformer_seed99 (640394 params)
evaluated stead_attention_seed99 (111370 params)
evaluated pretrained_None (78090 params)
evaluated pretrained_Linear (86410 params)
evaluated pretrained_LSTM (178442 params)
evaluated pretrained_attention (111370 params)
evaluated pretrained_Transformer (640394 params)
baseline baseline_identity
baseline baseline_bandpass
failed models: [] | compat shim used for: []


In [ ]:
E1_REF = {"None": (0.9483, 0.8765, 0.0708), "Linear": (0.9467, 0.8782, 0.0695), "LSTM": (0.9739, 0.9262, 0.0443),
          "attention": (0.9756, 0.9254, 0.0439), "Transformer": (0.9537, 0.8857, 0.0663)}

class Clusters:
    """Resample whole earthquakes (source_id). Vectorised for means; a loop for medians."""
    def __init__(self, ids, n_boot, seed):
        self.u, self.inv = np.unique(ids, return_inverse=True); self.K = len(self.u); self.n_boot = n_boot
        rng = np.random.default_rng(seed)
        self.W = np.stack([np.bincount(rng.integers(0, self.K, self.K), minlength=self.K) for _ in range(n_boot)]).astype(float)
        self.members = [np.where(self.inv == k)[0] for k in range(self.K)]; self.rng = np.random.default_rng(seed + 1)
    def sums(self, v, mask=None):
        v = np.asarray(v, float); m = np.ones(len(v), bool) if mask is None else mask
        S = np.bincount(self.inv[m], weights=np.nan_to_num(v[m]), minlength=self.K); Nn = np.bincount(self.inv[m], minlength=self.K).astype(float)
        return S, Nn
    def mean_ci(self, v, mask=None):
        S, Nn = self.sums(v, mask); tot = Nn.sum()
        if tot == 0: return np.nan, np.nan, np.nan
        est = S.sum() / tot; den = self.W @ Nn
        with np.errstate(invalid="ignore", divide="ignore"): bs = (self.W @ S) / den
        bs = bs[np.isfinite(bs)]
        return float(est), float(np.quantile(bs, .025)), float(np.quantile(bs, .975))
    def diff_ci(self, a, b, mask=None):
        return self.mean_ci(np.asarray(a, float) - np.asarray(b, float), mask)
    def median_ci(self, v2d, n=300):
        """v2d (N, 3): pooled median over trace-components, resampling earthquakes."""
        est = float(np.median(v2d)); out = []
        for _ in range(n):
            pick = self.rng.integers(0, self.K, self.K); ii = np.concatenate([self.members[k] for k in pick])
            out.append(np.median(v2d[ii]))
        return est, float(np.quantile(out, .025)), float(np.quantile(out, .975))

def load_ps(mid):
    return read_csv_keep(RESULTS/f"per_sample_{mid}.csv.gz")

TAB = {}
if RUN_EVAL:
    REG = load_registry(); ids = [k for k, v in REG.items() if v.get("status") == "ok" and (RESULTS/f"per_sample_{k}.csv.gz").exists()]
    CL = Clusters(MTEST.source_id.to_numpy(), N_BOOT, BOOT_SEED); PS = {k: load_ps(k) for k in ids}
    pool = lambda d, p: d[[f"{p}_c0", f"{p}_c1", f"{p}_c2"]].to_numpy()
    rows = []
    for k in ids:
        d = PS[k]; r = REG[k]; s = summarise_model(d)
        me, mlo, mhi = CL.median_ci(pool(d, "ev_eq")); ne, nlo, nhi = CL.median_ci(pool(d, "ev_noise"))
        ce, clo, chi = CL.mean_ci(d.ev_eq_mean_clipped.to_numpy()); cn, cnlo, cnhi = CL.mean_ci(d.ev_noise_mean_clipped.to_numpy())
        lo_, lo_lo, lo_hi = CL.mean_ci(d.mse_total.to_numpy())
        rows.append(dict(model_id=k, kind=r["kind"], bottleneck=r["bottleneck"], seed=r.get("seed", -1), n_params=r.get("n_params"),
                         median_ev_eq=s["median_ev_eq"], median_ev_eq_lo=mlo, median_ev_eq_hi=mhi,
                         median_ev_noise=s["median_ev_noise"], median_ev_noise_lo=nlo, median_ev_noise_hi=nhi,
                         mean_ev_eq_clipped=ce, mean_ev_eq_clipped_lo=clo, mean_ev_eq_clipped_hi=chi,
                         mean_ev_noise_clipped=cn, mean_ev_noise_clipped_lo=cnlo, mean_ev_noise_clipped_hi=cnhi,
                         frac_ev_eq_lt0=s["frac_ev_eq_lt0"], frac_ev_noise_lt0=s["frac_ev_noise_lt0"],
                         test_loss=lo_, test_loss_lo=lo_lo, test_loss_hi=lo_hi, mean_corr_eq=s["mean_corr_eq"], runtime_s=r.get("runtime_s")))
    SUM = pd.DataFrame(rows); SUM.to_csv(RESULTS/"table_instance_eval_per_model.csv", index=False); TAB["summary"] = SUM
    print("INSTANCE test split, pooled median EV with 95 % CI over earthquakes (28 earthquakes in the test split):")
    print(SUM[["model_id", "median_ev_eq", "median_ev_eq_lo", "median_ev_eq_hi", "median_ev_noise", "test_loss", "n_params"]].round(4).to_string(index=False))

    cmp_rows = []
    for b in BOTTLENECKS:
        row = dict(bottleneck=b, E1_stead_on_stead_eq=E1_REF[b][0], E1_noise=E1_REF[b][1], E1_test_loss=E1_REF[b][2])
        for tag, key in (("E2_stead_on_instance", f"stead_{b}_seed99"), ("E3_instance_on_instance", f"instance_{b}_seed99")):
            s = SUM[SUM.model_id == key]
            row[f"{tag}_eq"] = float(s.median_ev_eq.iloc[0]) if len(s) else np.nan
            row[f"{tag}_noise"] = float(s.median_ev_noise.iloc[0]) if len(s) else np.nan
            row[f"{tag}_test_loss"] = float(s.test_loss.iloc[0]) if len(s) else np.nan
        cmp_rows.append(row)
    CMP = pd.DataFrame(cmp_rows); CMP.to_csv(RESULTS/"table_E1_E2_E3.csv", index=False); TAB["cmp"] = CMP
    print("\nE1 / E2 / E3, pooled median EV (E1 values are from Phase 1, a different test set, one seed):"); print(CMP.round(4).to_string(index=False))

    pairs = []
    sid = lambda on, b: f"{on}_{b}_seed99"
    for b in BOTTLENECKS: pairs.append((sid("instance", b), sid("stead", b), "E3 vs E2, same bottleneck"))
    for i in range(len(BOTTLENECKS)):
        for j in range(i + 1, len(BOTTLENECKS)):
            pairs.append((sid("stead", BOTTLENECKS[i]), sid("stead", BOTTLENECKS[j]), "E2 bottleneck vs bottleneck"))
            pairs.append((sid("instance", BOTTLENECKS[i]), sid("instance", BOTTLENECKS[j]), "E3 bottleneck vs bottleneck"))
    prow = []
    for a, b_, why in pairs:
        if a in PS and b_ in PS:
            for kind, col in (("eq", "ev_eq_mean_clipped"), ("noise", "ev_noise_mean_clipped")):
                m_, lo, hi = CL.diff_ci(PS[a][col].to_numpy(), PS[b_][col].to_numpy())
                prow.append(dict(A=a, B=b_, comparison=why, branch=kind, mean_diff_A_minus_B=m_, ci_lo=lo, ci_hi=hi, significant_at_95=bool(lo > 0 or hi < 0),
                                 note="CI over earthquakes in the test split; one training seed only, so training randomness is NOT included"))
    pd.DataFrame(prow).to_csv(RESULTS/"table_instance_paired.csv", index=False)

    grp = []
    for k in ids:
        d = PS[k]
        for g, mask in [("all", np.ones(len(d), bool)), ("energy_ge1pct", (d.quake_energy_frac >= 0.01).to_numpy())] + \
                       [(f"channel_{c}", (d.station_channels == c).to_numpy()) for c in sorted(d.station_channels.unique())]:
            for kind, col in (("eq", "ev_eq_mean_clipped"), ("noise", "ev_noise_mean_clipped")):
                m_, lo, hi = CL.mean_ci(d[col].to_numpy(), mask)
                grp.append(dict(model_id=k, group=g, branch=kind, n=int(mask.sum()), mean_ev_clipped=m_, ci_lo=lo, ci_hi=hi))
    pd.DataFrame(grp).to_csv(RESULTS/"table_instance_by_group.csv", index=False)

    edges = snr_edges(0.1, 100, SNR_BINS_PER_DECADE); b_ = bin_index(MTEST.snr.to_numpy(), edges); rows = []
    for k in ids:
        d = PS[k]
        for gname, gm in (("all", np.ones(len(d), bool)), ("energy_ge1pct", (d.quake_energy_frac >= 0.01).to_numpy())):
            for kind, col in (("eq", "ev_eq_mean_clipped"), ("noise", "ev_noise_mean_clipped")):
                for j in range(len(edges) - 1):
                    mk = gm & (b_ == j); m_, lo, hi = CL.mean_ci(d[col].to_numpy(), mk)
                    rows.append(dict(model_id=k, ev_kind=kind, group=gname, bin_lo=edges[j], bin_hi=edges[j + 1], n=int(mk.sum()), mean_ev_clipped=m_, ci_lo=lo, ci_hi=hi))
    pd.DataFrame(rows).to_csv(RESULTS/"table_instance_ev_vs_snr.csv", index=False)
    log_entry("E2/E3 tables", "cluster-bootstrap tables over earthquakes", f"{len(ids)} models")
else:
    print("tables skipped")

INSTANCE test split, pooled median EV with 95 % CI over earthquakes (28 earthquakes in the test split):
                   model_id  median_ev_eq  median_ev_eq_lo  median_ev_eq_hi  median_ev_noise  test_loss  n_params
       instance_LSTM_seed99        0.9220           0.9054           0.9368           0.8940     0.0613    178442
     instance_Linear_seed99        0.8871           0.8512           0.9137           0.8581     0.0903     86410
       instance_None_seed99        0.8927           0.8549           0.9202           0.8508     0.0886     78090
instance_Transformer_seed99        0.8975           0.8655           0.9224           0.8638     0.0847    640394
  instance_attention_seed99        0.9216           0.9087           0.9358           0.8912     0.0639    111370
          stead_LSTM_seed99        0.9562           0.9482           0.9658           0.9133     0.0527    178442
        stead_Linear_seed99        0.9139           0.8865           0.9348           0.8276     0

In [9]:
if RUN_EVAL:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    SUM = TAB["summary"]; order = [m for m in ["baseline_identity", "baseline_bandpass"] if m in set(SUM.model_id)]
    groups = [("stead_", "E2: trained on STEAD"), ("instance_", "E3: trained on INSTANCE"), ("pretrained_", "authors' weights")]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for a, (col, title) in zip(ax, (("median_ev_eq", "earthquake branch"), ("median_ev_noise", "noise branch"))):
        w = 0.27
        for gi, (pre, lab) in enumerate(groups):
            sub = SUM[SUM.model_id.str.startswith(pre)].set_index("bottleneck").reindex(BOTTLENECKS)
            x = np.arange(len(BOTTLENECKS)) + (gi - 1) * w
            err = np.vstack([sub[col] - sub[col + "_lo"], sub[col + "_hi"] - sub[col]]) if col + "_lo" in sub else None
            a.bar(x, sub[col].to_numpy(), w, label=lab, yerr=err, capsize=2)
        for m in order:
            a.axhline(float(SUM[SUM.model_id == m][col].iloc[0]), ls="--", lw=.8, color="k" if "identity" in m else "gray", label=m)
        a.set_xticks(range(len(BOTTLENECKS))); a.set_xticklabels(BOTTLENECKS); a.set_ylabel("pooled median EV"); a.set_title(f"INSTANCE test split: {title}")
        a.set_ylim(min(0, np.nanmin(SUM[col]) - .05), 1.02)
    ax[0].legend(fontsize=7); plt.tight_layout(); save_fig(fig, "fig_instance_ev_per_model"); plt.show()

    T = pd.read_csv(RESULTS/"table_instance_ev_vs_snr.csv"); T = T[(T.group == "all")]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for a, kind in zip(ax, ("eq", "noise")):
        for k in sorted(set(T.model_id)):
            if k.startswith("pretrained_"): continue
            d = T[(T.model_id == k) & (T.ev_kind == kind)]; xc = np.sqrt(d.bin_lo * d.bin_hi)
            a.plot(xc, d.mean_ev_clipped, "-o" if k.startswith("instance_") else "--", ms=3, lw=1, label=k)
        a.set_xscale("log"); a.set_xlabel("input SNR (linear amplitude factor)"); a.set_ylabel("mean EV (clipped at -1)"); a.set_title(kind)
    ax[0].legend(fontsize=6, ncol=2); plt.tight_layout(); save_fig(fig, "fig_instance_ev_vs_snr"); plt.show()
else:
    print("figures skipped")

In [10]:
# good / average / failure examples for one reference model, test split
if RUN_EVAL:
    REG = load_registry(); okids = [k for k, v in REG.items() if v.get("status") == "ok"]
    ref = next((k for k in ("instance_attention_seed99", "instance_None_seed99", "stead_attention_seed99", "stead_None_seed99") if k in okids), None)
    if ref is None:
        print("no reference model for examples")
    else:
        df = PS[ref]; v = df.ev_eq_mean_clipped.to_numpy(); sig = df.quake_energy_frac.to_numpy() >= 0.01
        order_ = np.argsort(v); nfail = max(1, int(FAIL_FRACTION * len(df)))
        df.iloc[order_[:nfail]].to_csv(RESULTS/f"failure_cases_{ref}.csv", index=False)
        json.dump(dict(model=ref, n_failures=nfail, frac_energy_lt_1pct=float((df.quake_energy_frac.iloc[order_[:nfail]] < 0.01).mean()),
                       test_frac_energy_lt_1pct=float((~sig).mean()), median_snr_failures=float(df.snr.iloc[order_[:nfail]].median())),
                  open(RESULTS/"failure_cases_instance_summary.json", "w"), indent=2)
        rng = np.random.default_rng(EXAMPLE_SEED); med = np.median(v[sig])
        pick = {"good": rng.choice(np.where(sig & (v >= np.quantile(v[sig], .9)))[0], 1)[0],
                "average": rng.choice(np.where(sig)[0][np.argsort(np.abs(v[sig] - med))[:100]], 1)[0],
                "failure": rng.choice(order_[:nfail], 1)[0]}
        torch, _ = prepare_torch(); dev = get_device(); mdl = load_model(REG[ref]["model_path"], dev); check_forward(mdl, dev)
        ii = np.array(list(pick.values())); pe, pn = predict(mdl, np.ascontiguousarray(Xt[ii]), dev); tt_ = np.arange(600) / 10
        fig, ax = plt.subplots(3, 3, figsize=(14, 7), sharex=True)
        for c, (nm, i) in enumerate(pick.items()):
            x, y = Xt[i][2], Yt[i][2]
            ax[0, c].plot(tt_, x, lw=.6); ax[0, c].set_title(f"{nm}: row {i}, EV {v[i]:.2f}, snr {df.snr.iloc[i]:.2f}, {df.station_channels.iloc[i]}", fontsize=8)
            ax[1, c].plot(tt_, y, lw=.7, label="truth"); ax[1, c].plot(tt_, pe[c][2], lw=.7, label="model")
            ax[2, c].plot(tt_, x - y, lw=.7, label="truth"); ax[2, c].plot(tt_, pn[c][2], lw=.7, label="model")
        for r_, t_ in enumerate(("input (axis 3)", "earthquake", "noise")): ax[r_, 0].set_ylabel(t_)
        ax[1, 0].legend(fontsize=7); ax[2, 1].set_xlabel("s"); plt.tight_layout(); save_fig(fig, f"fig_instance_examples_{ref}"); plt.show()
        print("reference model for the examples:", ref)

reference model for the examples: instance_attention_seed99
